# NYC Taxi Big Data – Hadoop, Hive and Spark on Google Colab


## Step 1 – Install Java 8 and download Hadoop and Hive (about 5 minutes)

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y openjdk-8-jdk-headless > /dev/null
!wget -q -nc https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!wget -q -nc https://archive.apache.org/dist/hive/hive-3.1.3/apache-hive-3.1.3-bin.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz -C /opt && tar -xzf apache-hive-3.1.3-bin.tar.gz -C /opt
!mv -n /opt/hadoop-3.3.6 /opt/hadoop ; mv -n /opt/apache-hive-3.1.3-bin /opt/hive
!pip -q install pyspark==3.5.3 kagglehub openpyxl
print("Downloads finished")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## Step 2 – Set environment variables

In [ ]:
import os
env = {
  "JAVA_HOME": "/usr/lib/jvm/java-8-openjdk-amd64",
  "HADOOP_HOME": "/opt/hadoop", "HADOOP_CONF_DIR": "/opt/hadoop/etc/hadoop",
  "YARN_CONF_DIR": "/opt/hadoop/etc/hadoop", "HADOOP_MAPRED_HOME": "/opt/hadoop",
  "HIVE_HOME": "/opt/hive", "PYSPARK_PYTHON": "python3",
  # Colab runs as root, so Hadoop needs to be told which user runs each daemon
  "HDFS_NAMENODE_USER": "root", "HDFS_DATANODE_USER": "root", "HDFS_SECONDARYNAMENODE_USER": "root",
  "YARN_RESOURCEMANAGER_USER": "root", "YARN_NODEMANAGER_USER": "root", "MAPRED_HISTORYSERVER_USER": "root",
}
os.environ.update(env)
os.environ["PATH"] = "/usr/lib/jvm/java-8-openjdk-amd64/bin:/opt/hadoop/bin:/opt/hadoop/sbin:/opt/hive/bin:" + os.environ["PATH"]
with open("/opt/hadoop/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64\n")
!java -version
!hadoop version | head -1

openjdk version "1.8.0_504"
OpenJDK Runtime Environment (build 1.8.0_504-8u504-ga-1ubuntu1~24.04.3-b01)
OpenJDK 64-Bit Server VM (build 25.504-b01, mixed mode)
Hadoop 3.3.6


## Step 3 – Hadoop configuration files (128 MB blocks, replication 1, checkpoint every 10 minutes)

In [ ]:
%%writefile /opt/hadoop/etc/hadoop/core-site.xml
<?xml version="1.0"?>
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
  <property><name>hadoop.tmp.dir</name><value>/root/hadoopdata/tmp</value></property>
  <property><name>hadoop.proxyuser.root.hosts</name><value>*</value></property>
  <property><name>hadoop.proxyuser.root.groups</name><value>*</value></property>
</configuration>

Overwriting /opt/hadoop/etc/hadoop/core-site.xml


In [ ]:
%%writefile /opt/hadoop/etc/hadoop/hdfs-site.xml
<?xml version="1.0"?>
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.blocksize</name><value>134217728</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///root/hadoopdata/namenode</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///root/hadoopdata/datanode</value></property>
  <property><name>dfs.namenode.checkpoint.dir</name><value>file:///root/hadoopdata/namesecondary</value></property>
  <property><name>dfs.namenode.checkpoint.period</name><value>600</value></property>
  <property><name>dfs.namenode.checkpoint.txns</name><value>100000</value></property>
  <property><name>dfs.namenode.secondary.http-address</name><value>localhost:9868</value></property>
  <property><name>dfs.permissions.enabled</name><value>false</value></property>
</configuration>

Overwriting /opt/hadoop/etc/hadoop/hdfs-site.xml


In [ ]:
%%writefile /opt/hadoop/etc/hadoop/mapred-site.xml
<?xml version="1.0"?>
<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>mapreduce.application.classpath</name><value>/opt/hadoop/share/hadoop/mapreduce/*:/opt/hadoop/share/hadoop/mapreduce/lib/*</value></property>
  <property><name>yarn.app.mapreduce.am.env</name><value>HADOOP_MAPRED_HOME=/opt/hadoop</value></property>
  <property><name>mapreduce.map.env</name><value>HADOOP_MAPRED_HOME=/opt/hadoop</value></property>
  <property><name>mapreduce.reduce.env</name><value>HADOOP_MAPRED_HOME=/opt/hadoop</value></property>
  <property><name>mapreduce.map.memory.mb</name><value>1024</value></property>
  <property><name>mapreduce.reduce.memory.mb</name><value>1024</value></property>
  <property><name>mapreduce.map.java.opts</name><value>-Xmx819m</value></property>
  <property><name>mapreduce.reduce.java.opts</name><value>-Xmx819m</value></property>
  <property><name>yarn.app.mapreduce.am.resource.mb</name><value>1024</value></property>
  <property><name>mapreduce.jobhistory.address</name><value>localhost:10020</value></property>
  <property><name>mapreduce.jobhistory.webapp.address</name><value>localhost:19888</value></property>
</configuration>

Overwriting /opt/hadoop/etc/hadoop/mapred-site.xml


In [ ]:
%%writefile /opt/hadoop/etc/hadoop/yarn-site.xml
<?xml version="1.0"?>
<configuration>
  <property><name>yarn.resourcemanager.hostname</name><value>localhost</value></property>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
  <property><name>yarn.nodemanager.resource.memory-mb</name><value>8192</value></property>
  <property><name>yarn.nodemanager.resource.cpu-vcores</name><value>2</value></property>
  <property><name>yarn.scheduler.minimum-allocation-mb</name><value>512</value></property>
  <property><name>yarn.scheduler.maximum-allocation-mb</name><value>4096</value></property>
  <property><name>yarn.nodemanager.vmem-check-enabled</name><value>false</value></property>
  <property><name>yarn.nodemanager.pmem-check-enabled</name><value>false</value></property>
  <property><name>yarn.nodemanager.disk-health-checker.max-disk-utilization-per-disk-percentage</name><value>99</value></property>
  <property><name>yarn.nodemanager.env-whitelist</name><value>JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME</value></property>
</configuration>

Overwriting /opt/hadoop/etc/hadoop/yarn-site.xml


## Step 4 – Format the NameNode and start all daemons

In [ ]:
!mkdir -p /root/hadoopdata/namenode /root/hadoopdata/datanode /root/hadoopdata/namesecondary /root/hadoopdata/tmp
!hdfs namenode -format -force -nonInteractive 2>&1 | grep -iE "formatted|error"
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!hdfs --daemon start secondarynamenode
!yarn --daemon start resourcemanager
!yarn --daemon start nodemanager
!mapred --daemon start historyserver
import time; time.sleep(15)
print("Daemons started")

2026-09-21 19:57:43,027 INFO common.Storage: Storage directory /root/hadoopdata/namenode has been successfully formatted.
Daemons started


### 📸 Screenshot 1 – running daemons (`jps`)
You should see **NameNode, DataNode, SecondaryNameNode, ResourceManager, NodeManager, JobHistoryServer**.

In [ ]:
!jps

5747 DataNode
5927 NodeManager
5992 JobHistoryServer
5694 NameNode
5871 ResourceManager
6239 Jps
5807 SecondaryNameNode


In [ ]:
!hdfs dfsadmin -report | head -20
!yarn node -list

Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 89736818688 (83.57 GB)
DFS Remaining: 89736794112 (83.57 GB)
DFS Used: 24576 (24 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
2026-09-21 19:58:33,775 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at localhost/127.0.0.1:8032
Total Nodes:1
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
d21b880bee1d:40953	        RUNNING	d21b880bee1d:8042	                           0


### Optional 📸 – web interfaces
Run the cell and click the links that appear. They open the real NameNode UI (:9870), YARN UI (:8088) and Secondary NameNode UI (:9868) in a new browser tab.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(9870, path="/dfshealth.html#tab-overview", anchor_text="NameNode UI (9870)")
output.serve_kernel_port_as_window(8088, path="/cluster", anchor_text="YARN ResourceManager UI (8088)")
output.serve_kernel_port_as_window(9868, path="/status.html", anchor_text="Secondary NameNode UI (9868)")

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Step 5 – Get the data
This downloads the **full** January 2015 CSV from the same Kaggle dataset (so HDFS gets 3–4+ blocks). It also recreates your **Excel extract** (the first 1,048,575 trips), which is what the report's analysis used.

If the download asks for login: in Kaggle go to *Settings → API → Create New Token*, upload the `kaggle.json` file with the Files panel on the left, and run the cell again.

In [ ]:
import os, glob, shutil, kagglehub
if os.path.exists("kaggle.json"):
    os.makedirs("/root/.kaggle", exist_ok=True); shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
os.makedirs("/content/data", exist_ok=True)
try:
    p = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data", path="yellow_tripdata_2015-01.csv")
except Exception as e:
    print("Single-file download failed (", e, ") - downloading the whole dataset instead...")
    d = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")
    p = glob.glob(d + "/**/yellow_tripdata_2015-01.csv", recursive=True)[0]
shutil.copy(p, "/content/data/yellow_tripdata_2015-01.csv")
print("Full file ready:", p)

100%|██████████| 504M/504M [00:03<00:00, 158MB/s]

Extracting zip of yellow_tripdata_2015-01.csv...


Full file ready: /root/.cache/kagglehub/datasets/elemento/nyc-yellow-taxi-trip-data/versions/2/yellow_tripdata_2015-01.csv


**Excel extract:** drag your `taxi_2.xltx` into the Files panel on the left (the folder icon) before running the next cell. The cell then converts exactly your file, which takes about 3 minutes. If the file isn't there, it uses the first 1,048,575 trips of the full CSV instead.

In [ ]:
import csv, datetime, os, subprocess
out = "/content/data/yellow_tripdata_2015-01_extract.csv"
if os.path.exists("/content/taxi_2.xltx"):
    import openpyxl
    ws = openpyxl.load_workbook("/content/taxi_2.xltx", read_only=True).worksheets[0]
    with open(out, "w", newline="") as f:
        w = csv.writer(f)
        for r in ws.iter_rows(values_only=True):
            w.writerow([v.strftime("%Y-%m-%d %H:%M:%S") if isinstance(v, datetime.datetime) else v for v in r])
    print("Converted taxi_2.xltx ->", out)
else:
    subprocess.run(f"head -n 1048576 /content/data/yellow_tripdata_2015-01.csv > {out}", shell=True)
    print("taxi_2.xltx not found - used the first 1,048,575 trips of the full CSV")

taxi_2.xltx not found - used the first 1,048,575 trips of the full CSV


### 📸 Screenshot 2 – the local files (size, row count, columns)

In [ ]:
!ls -l /content/data/
!wc -l /content/data/*.csv
!head -3 /content/data/yellow_tripdata_2015-01.csv
import os
for f in ["yellow_tripdata_2015-01.csv", "yellow_tripdata_2015-01_extract.csv"]:
    s = os.path.getsize("/content/data/" + f)
    print(f"{f}: {s:,} bytes -> {-(-s // 134217728)} blocks of 128 MB")

total 2098924
-rw-r--r-- 1 root root 1985964692 Sep 21 19:59 yellow_tripdata_2015-01.csv
-rw-r--r-- 1 root root  163326797 Sep 21 19:59 yellow_tripdata_2015-01_extract.csv
  12748987 /content/data/yellow_tripdata_2015-01.csv
   1048576 /content/data/yellow_tripdata_2015-01_extract.csv
  13797563 total
VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896484375,40.750110626220703,1,N,-73.974784851074219,40.750617980957031,1,12,1,0.5,3.25,0,0.3,17.05
1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.30,-74.00164794921875,40.7242431640625,1,N,-73.994415283203125,40.759109497070313,1,14.5,0.5,0.5,2,0,0.3,17.8
yellow_tripdata_2015-01.csv: 1,985,964,692 bytes -> 15 blocks of 128 MB
yellow_tripdata_2015-01_extract.csv

## Step 6 – Ingest into HDFS

### 📸 Screenshot 3 – upload and listing

In [ ]:
!hdfs dfs -mkdir -p /user/hadoop/nyctaxi/raw /user/hadoop/nyctaxi/extract
!time hdfs dfs -put -f /content/data/yellow_tripdata_2015-01.csv /user/hadoop/nyctaxi/raw/
!hdfs dfs -put -f /content/data/yellow_tripdata_2015-01_extract.csv /user/hadoop/nyctaxi/extract/
!hdfs dfs -ls -h /user/hadoop/nyctaxi/raw /user/hadoop/nyctaxi/extract
!hdfs dfs -du -h /user/hadoop/nyctaxi


real	0m23.391s
user	0m10.497s
sys	0m2.540s
Found 1 items
-rw-r--r--   1 root supergroup      1.8 G 2026-09-21 20:00 /user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv
Found 1 items
-rw-r--r--   1 root supergroup    155.8 M 2026-09-21 20:00 /user/hadoop/nyctaxi/extract/yellow_tripdata_2015-01_extract.csv
155.8 M  155.8 M  /user/hadoop/nyctaxi/extract
1.8 G    1.8 G    /user/hadoop/nyctaxi/raw


### 📸 Screenshot 4 – block distribution and replication (`fsck`)
Note **Total blocks** for the full file. This goes into the report.

In [ ]:
!hdfs fsck /user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv -files -blocks -locations 2>/dev/null | grep -vE "^Connecting|^FSCK started" | head -40
!hdfs fsck /user/hadoop/nyctaxi -files -blocks 2>/dev/null | grep -E "Total size|Total files|Total blocks|Average block replication|Under-replicated|Default replication|Status"
!hdfs dfs -stat "name=%n  size=%b  blocksize=%o  replication=%r" /user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv


/user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv 1985964692 bytes, replicated: replication=1, 15 block(s):  OK
0. BP-1054428040-172.28.0.12-1790020662976:blk_1073741825_1001 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-363cdd4e-60d7-477e-8854-d72ee2c1f6b2,DISK]]
1. BP-1054428040-172.28.0.12-1790020662976:blk_1073741826_1002 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-363cdd4e-60d7-477e-8854-d72ee2c1f6b2,DISK]]
2. BP-1054428040-172.28.0.12-1790020662976:blk_1073741827_1003 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-363cdd4e-60d7-477e-8854-d72ee2c1f6b2,DISK]]
3. BP-1054428040-172.28.0.12-1790020662976:blk_1073741828_1004 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-363cdd4e-60d7-477e-8854-d72ee2c1f6b2,DISK]]
4. BP-1054428040-172.28.0.12-1790020662976:blk_1073741829_1005 len=134217728 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-363cdd4e-60d7-477e-8854-d72ee2c1f6b2,DISK]]

### 📸 Screenshot 5 – replication demo (one DataNode cannot hold 3 copies)

In [ ]:
!hdfs dfs -setrep 3 /user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv
!hdfs fsck /user/hadoop/nyctaxi/raw 2>/dev/null | grep -iE "Under-replicated|Average block replication|Total blocks|Missing replicas"
!hdfs dfs -setrep 1 /user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv > /dev/null

Replication 3 set: /user/hadoop/nyctaxi/raw/yellow_tripdata_2015-01.csv
 Total blocks (validated):	15 (avg. block size 132397646 B)
 Under-replicated blocks:	15 (100.0 %)
 Average block replication:	1.0
 Missing replicas:		30 (66.666664 %)


### 📸 Screenshot 6 – FSImage, EditLog and checkpoint
Compare the `fsimage_…` numbers before and after `saveNamespace`: a new FSImage with a higher transaction ID appears.

In [ ]:
print("=== BEFORE checkpoint ===")
!ls -l /root/hadoopdata/namenode/current | grep -E "fsimage|edits_inprogress|seen_txid"
!hdfs dfsadmin -safemode enter
!hdfs dfsadmin -saveNamespace
!hdfs dfsadmin -safemode leave
print("=== AFTER checkpoint ===")
!ls -l /root/hadoopdata/namenode/current | grep -E "fsimage|edits_inprogress|seen_txid"

=== BEFORE checkpoint ===
-rw-r--r-- 1 root root 1048576 Sep 21 20:01 edits_inprogress_0000000000000000010
-rw-r--r-- 1 root root     399 Sep 21 19:57 fsimage_0000000000000000000
-rw-r--r-- 1 root root      62 Sep 21 19:57 fsimage_0000000000000000000.md5
-rw-r--r-- 1 root root       3 Sep 21 19:59 seen_txid
Safe mode is ON
Save namespace successful
Safe mode is OFF
=== AFTER checkpoint ===
-rw-r--r-- 1 root root 1048576 Sep 21 20:01 edits_inprogress_0000000000000000076
-rw-r--r-- 1 root root     399 Sep 21 19:57 fsimage_0000000000000000000
-rw-r--r-- 1 root root      62 Sep 21 19:57 fsimage_0000000000000000000.md5
-rw-r--r-- 1 root root    1576 Sep 21 20:01 fsimage_0000000000000000075
-rw-r--r-- 1 root root      62 Sep 21 20:01 fsimage_0000000000000000075.md5
-rw-r--r-- 1 root root       3 Sep 21 20:01 seen_txid


## Step 7 – Hive on MapReduce

In [ ]:
# Hive 3.1.3 vs Hadoop 3.3.6 guava clash fix, then create the metastore
!rm -f /opt/hive/lib/guava-19.0.jar && cp /opt/hadoop/share/hadoop/common/lib/guava-27.0-jre.jar /opt/hive/lib/
!hdfs dfs -mkdir -p /user/hive/warehouse /tmp && hdfs dfs -chmod -R 777 /tmp /user/hive/warehouse
%cd /content
!schematool -dbType derby -initSchema 2>&1 | tail -3

/content

Initialization script completed
schemaTool completed


In [ ]:
%%writefile /content/03_hive_mapreduce.hql
-- =====================================================================
-- 03 - Distributed processing with Hive on MapReduce (Task 3)
-- Run:   cd ~ && hive -f ~/nyc_taxi_bigdata/scripts/03_hive_mapreduce.hql 2>&1 | tee ~/hive_run.log
-- Or paste each block into the Hue Hive editor.
-- Every SELECT below is compiled by Hive into MapReduce job(s) and
-- submitted to YARN. Watch them at http://localhost:8088 (SCREENSHOT).
-- Hive prints "Time taken: N seconds" after each query: record these
-- numbers for the Hive vs Spark comparison table.
-- =====================================================================

SET hive.execution.engine=mr;            -- force classic MapReduce
SET hive.cli.print.header=true;
SET mapreduce.job.reduces=-1;            -- let Hive choose #reducers
SET hive.exec.reducers.bytes.per.reducer=268435456;

CREATE DATABASE IF NOT EXISTS nyctaxi;
USE nyctaxi;

-- ---------------------------------------------------------------------
-- 1. External table over the raw CSV already in HDFS (schema-on-read).
--    Dropping this table does NOT delete the file in HDFS.
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS trips_raw;
CREATE EXTERNAL TABLE trips_raw (
  vendorid              INT,
  tpep_pickup_datetime  STRING,
  tpep_dropoff_datetime STRING,
  passenger_count       INT,
  trip_distance         DOUBLE,
  pickup_longitude      DOUBLE,
  pickup_latitude       DOUBLE,
  ratecodeid            INT,
  store_and_fwd_flag    STRING,
  dropoff_longitude     DOUBLE,
  dropoff_latitude      DOUBLE,
  payment_type          INT,
  fare_amount           DOUBLE,
  extra                 DOUBLE,
  mta_tax               DOUBLE,
  tip_amount            DOUBLE,
  tolls_amount          DOUBLE,
  improvement_surcharge DOUBLE,
  total_amount          DOUBLE
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/user/hadoop/nyctaxi/extract'
TBLPROPERTIES ('skip.header.line.count'='1');

-- Q0. Row count  (1 map task per HDFS block -> 1 reducer)
SELECT COUNT(*) AS total_rows FROM trips_raw;

-- ---------------------------------------------------------------------
-- 2. Cleaning (map-only job) -> columnar ORC table
--    Rules: valid NYC coordinates, 0 < distance <= 100 miles,
--    fare 2.50-500 USD, duration 1-180 min, 1-6 passengers.
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS trips_clean;
CREATE TABLE trips_clean STORED AS ORC AS
SELECT *
FROM (
  SELECT
    vendorid,
    CAST(tpep_pickup_datetime AS TIMESTAMP)                     AS pickup_ts,
    hour(tpep_pickup_datetime)                                  AS pickup_hour,
    -- 1 = Sunday ... 7 = Saturday (1900-01-07 was a Sunday)
    pmod(datediff(to_date(tpep_pickup_datetime), '1900-01-07'), 7) + 1 AS pickup_dow,
    to_date(tpep_pickup_datetime)                               AS pickup_date,
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0
                                                                AS duration_min,
    passenger_count, trip_distance, ratecodeid, payment_type,
    pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude,
    fare_amount, tip_amount, tolls_amount, total_amount
  FROM trips_raw
) t
WHERE pickup_latitude  BETWEEN 40.49 AND 40.92
  AND pickup_longitude BETWEEN -74.27 AND -73.68
  AND trip_distance > 0 AND trip_distance <= 100
  AND fare_amount BETWEEN 2.5 AND 500
  AND duration_min BETWEEN 1 AND 180
  AND passenger_count BETWEEN 1 AND 6;

SELECT COUNT(*) AS clean_rows FROM trips_clean;

-- ---------------------------------------------------------------------
-- Q1. DEMAND BY HOUR  (the classic MapReduce "group by" pattern)
--   MAP    : each map task reads one 128 MB block, emits (pickup_hour, 1)
--   COMBINE: partial counts per hour inside each mapper (map-side aggr.)
--   SHUFFLE: all pairs with the same hour go to the same reducer
--   REDUCE : sums the counts and averages per hour
-- ---------------------------------------------------------------------
EXPLAIN
SELECT pickup_hour, COUNT(*) AS trips FROM trips_clean GROUP BY pickup_hour;

SELECT pickup_hour,
       COUNT(*)                       AS trips,
       ROUND(AVG(fare_amount), 2)     AS avg_fare,
       ROUND(AVG(duration_min), 2)    AS avg_duration_min,
       ROUND(AVG(trip_distance / (duration_min / 60.0)), 2) AS avg_speed_mph
FROM trips_clean
GROUP BY pickup_hour
ORDER BY pickup_hour;

-- Q2. DEMAND BY DAY-OF-WEEK x HOUR (heat-map input)
SELECT pickup_dow, pickup_hour, COUNT(*) AS trips
FROM trips_clean
GROUP BY pickup_dow, pickup_hour
ORDER BY pickup_dow, pickup_hour;

-- Q3. REVENUE AND TIPPING BY PAYMENT TYPE
--     1=Credit card 2=Cash 3=No charge 4=Dispute
SELECT payment_type,
       COUNT(*)                                           AS trips,
       ROUND(SUM(total_amount), 0)                        AS revenue_usd,
       ROUND(AVG(tip_amount), 2)                          AS avg_tip,
       ROUND(100 * AVG(tip_amount / fare_amount), 2)      AS avg_tip_pct
FROM trips_clean
GROUP BY payment_type
ORDER BY trips DESC;

-- Q4. TOP 20 PICKUP HOTSPOTS (~1 km grid cells) -> 2 MR jobs (group by, then global sort)
SELECT ROUND(pickup_latitude, 2)  AS lat_cell,
       ROUND(pickup_longitude, 2) AS lon_cell,
       COUNT(*)                   AS trips,
       ROUND(AVG(total_amount), 2) AS avg_total
FROM trips_clean
GROUP BY ROUND(pickup_latitude, 2), ROUND(pickup_longitude, 2)
ORDER BY trips DESC
LIMIT 20;

-- Q5. DAILY TREND (for the time-series chart / weather or holiday effects)
SELECT pickup_date, COUNT(*) AS trips, ROUND(SUM(total_amount), 0) AS revenue_usd
FROM trips_clean
GROUP BY pickup_date
ORDER BY pickup_date;

-- ---------------------------------------------------------------------
-- Optional: same Q1 on the RAW text table, to compare text vs ORC time
-- ---------------------------------------------------------------------
SELECT hour(tpep_pickup_datetime) AS h, COUNT(*) AS trips
FROM trips_raw GROUP BY hour(tpep_pickup_datetime) ORDER BY h;

Writing /content/03_hive_mapreduce.hql


### 📸 Screenshots 7–9 – Hive jobs
This takes about 10–20 minutes, because every query becomes a MapReduce job on YARN. While it runs, you can open the YARN UI link from Step 4 to watch the jobs.

Screenshot these parts of the output:
- **7**: one "Launching Job … Hadoop job information: number of mappers …; number of reducers …" block, including the progress lines;
- **8**: the `EXPLAIN` output (Map Operator Tree / Reduce Operator Tree);
- **9**: the results of Q1 (demand by hour).

In [ ]:
%cd /content
!hive -f /content/03_hive_mapreduce.hql 2>&1 | grep -vE "SLF4J|WARN|^Hive Session ID|^Logging initialized" | tee /content/hive_run.log

/content

Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
OK
Time taken: 1.226 seconds
OK
Time taken: 0.052 seconds
OK
Time taken: 0.127 seconds
OK
Time taken: 0.646 seconds
Query ID = root_20260921200204_fc40f594-20d1-4b01-bc1c-7a247a8f3eee
Total jobs = 1
Launching Job 1 out of 1
Number of reduce tasks determined at compile time: 1
In order to change the average load for a reducer (in bytes):
  set hive.exec.reducers.bytes.per.reducer=<number>
In order to limit the maximum number of reducers:
  set hive.exec.reducers.max=<number>
In order to set a constant number of reducers:
  set mapreduce.job.reduces=<number>
Starting Job = job_1790020707418_0001, Tracking URL = http://d21b880bee1d:8088/proxy/application_1790020707418_0001/
Kill Command = /opt/hadoop/bin/mapred job  -kill job_1790020707418_0001
Hadoop job information for Stage-1: number of mappers: 1; number

### 📸 Screenshot 10 – Hive execution times (goes into Table 8 of the report)

In [ ]:
!grep -E "Time taken" /content/hive_run.log

Time taken: 1.226 seconds
Time taken: 0.052 seconds
Time taken: 0.127 seconds
Time taken: 0.646 seconds
Time taken: 54.521 seconds, Fetched: 1 row(s)
Time taken: 0.121 seconds
Time taken: 72.106 seconds
Time taken: 0.242 seconds, Fetched: 1 row(s)
Time taken: 0.305 seconds, Fetched: 49 row(s)
Time taken: 83.064 seconds, Fetched: 24 row(s)
Time taken: 83.884 seconds, Fetched: 168 row(s)
Time taken: 83.712 seconds, Fetched: 4 row(s)
Time taken: 88.341 seconds, Fetched: 20 row(s)
Time taken: 78.979 seconds, Fetched: 31 row(s)
Time taken: 83.056 seconds, Fetched: 24 row(s)


## Step 8 – Spark on YARN (same queries, for the comparison)

In [ ]:
%%writefile /content/04_spark_analytics.py
"""
04 - Same analysis as the Hive script, but in Spark (DataFrame API + Spark SQL).
Used for Task 3 (Spark workflow + Hive-vs-Spark comparison) and to
produce the result files that the charts are drawn from (Task 4).

Submit to YARN:
  spark-submit --master yarn --deploy-mode client \
      --num-executors 2 --executor-cores 2 --executor-memory 2g --driver-memory 2g \
      ~/nyc_taxi_bigdata/scripts/04_spark_analytics.py \
      hdfs:///user/hadoop/nyctaxi/raw hdfs:///user/hadoop/nyctaxi

Test locally (no Hadoop):
  python3 04_spark_analytics.py file:///path/to/csv_folder file:///path/to/out local[*]
"""
import sys, time
from pyspark.sql import SparkSession, functions as F

RAW = sys.argv[1] if len(sys.argv) > 1 else "hdfs:///user/hadoop/nyctaxi/raw"
BASE = sys.argv[2] if len(sys.argv) > 2 else "hdfs:///user/hadoop/nyctaxi"
MASTER = sys.argv[3] if len(sys.argv) > 3 else None

b = (SparkSession.builder.appName("NYC-Taxi-Analytics")
     .config("spark.sql.session.timeZone", "UTC"))   # no DST shifts on timestamps
if MASTER:
    b = b.master(MASTER)
spark = b.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
timings = []


def timed(name, fn):
    """Run an ACTION and record wall-clock time (for the Hive vs Spark table)."""
    t0 = time.time()
    out = fn()
    secs = round(time.time() - t0, 2)
    timings.append((name, secs))
    print(f"[TIME] {name}: {secs} s")
    return out


# ---------- 1. Read (lazy TRANSFORMATION: nothing runs yet) ----------
schema = """VendorID INT, tpep_pickup_datetime STRING, tpep_dropoff_datetime STRING,
passenger_count INT, trip_distance DOUBLE, pickup_longitude DOUBLE, pickup_latitude DOUBLE,
RatecodeID INT, store_and_fwd_flag STRING, dropoff_longitude DOUBLE, dropoff_latitude DOUBLE,
payment_type INT, fare_amount DOUBLE, extra DOUBLE, mta_tax DOUBLE, tip_amount DOUBLE,
tolls_amount DOUBLE, improvement_surcharge DOUBLE, total_amount DOUBLE"""
raw = spark.read.csv(RAW, header=True, schema=schema)
print("Input partitions (≈ one per 128 MB HDFS block):", raw.rdd.getNumPartitions())

raw_rows = timed("Q0 count raw rows", raw.count)

# ---------- 2. Clean + feature columns (transformations) ----------
ts_fmt = "yyyy-MM-dd HH:mm:ss"
df = (raw
      .withColumn("pickup_ts", F.to_timestamp("tpep_pickup_datetime", ts_fmt))
      .withColumn("dropoff_ts", F.to_timestamp("tpep_dropoff_datetime", ts_fmt))
      .withColumn("duration_min",
                  (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60.0)
      .withColumn("pickup_hour", F.hour("pickup_ts"))
      .withColumn("pickup_dow", F.dayofweek("pickup_ts"))          # 1=Sun ... 7=Sat
      .withColumn("pickup_date", F.to_date("pickup_ts"))
      .filter(F.col("pickup_latitude").between(40.49, 40.92))
      .filter(F.col("pickup_longitude").between(-74.27, -73.68))
      .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100))
      .filter(F.col("fare_amount").between(2.5, 500))
      .filter(F.col("duration_min").between(1, 180))
      .filter(F.col("passenger_count").between(1, 6))
      .withColumn("speed_mph", F.col("trip_distance") / (F.col("duration_min") / 60.0))
      .drop("tpep_pickup_datetime", "tpep_dropoff_datetime", "store_and_fwd_flag"))

df = df.cache()                                   # keep in executor memory
clean_rows = timed("Q0b clean + count (cache fill)", df.count)
print(f"Raw rows: {raw_rows:,}  Clean rows: {clean_rows:,}  "
      f"Removed: {raw_rows - clean_rows:,} ({100 * (raw_rows - clean_rows) / raw_rows:.2f}%)")

# Save cleaned data as Parquet for the ML script (columnar + compressed)
timed("Write clean Parquet", lambda: df.write.mode("overwrite").parquet(f"{BASE}/clean_parquet"))


def save(d, name):
    d.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{BASE}/results/{name}")


# ---------- 3. Same questions as the Hive script ----------
# Q1 demand by hour  (map: partial count per partition -> shuffle by hour -> reduce: sum)
q1 = (df.groupBy("pickup_hour")
        .agg(F.count("*").alias("trips"),
             F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
             F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
             F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"))
        .orderBy("pickup_hour"))
timed("Q1 demand by hour", lambda: save(q1, "q1_hourly"))
q1.explain()            # physical plan: HashAggregate -> Exchange (shuffle) -> HashAggregate

# Q2 day-of-week x hour
q2 = df.groupBy("pickup_dow", "pickup_hour").count().withColumnRenamed("count", "trips") \
       .orderBy("pickup_dow", "pickup_hour")
timed("Q2 dow x hour", lambda: save(q2, "q2_dow_hour"))

# Q3 payment type (written in Spark SQL to show the SQL route too)
df.createOrReplaceTempView("trips_clean")
q3 = spark.sql("""
  SELECT payment_type, COUNT(*) AS trips, ROUND(SUM(total_amount),0) AS revenue_usd,
         ROUND(AVG(tip_amount),2) AS avg_tip,
         ROUND(100*AVG(tip_amount/fare_amount),2) AS avg_tip_pct
  FROM trips_clean GROUP BY payment_type ORDER BY trips DESC""")
timed("Q3 payment type", lambda: save(q3, "q3_payment"))

# Q4 hotspots (~1 km cells)
q4 = (df.groupBy(F.round("pickup_latitude", 2).alias("lat_cell"),
                 F.round("pickup_longitude", 2).alias("lon_cell"))
        .agg(F.count("*").alias("trips"), F.round(F.avg("total_amount"), 2).alias("avg_total"))
        .orderBy(F.desc("trips")))
timed("Q4 hotspots", lambda: save(q4.limit(500), "q4_hotspots"))

# Q5 daily trend
q5 = (df.groupBy("pickup_date")
        .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 0).alias("revenue_usd"))
        .orderBy("pickup_date"))
timed("Q5 daily trend", lambda: save(q5, "q5_daily"))

# Descriptive statistics for the report
desc = df.select("trip_distance", "duration_min", "fare_amount", "tip_amount",
                 "total_amount", "speed_mph", "passenger_count").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
timed("Summary statistics", lambda: save(desc, "summary_stats"))

save(spark.createDataFrame(
        timings + [("raw_rows", float(raw_rows)), ("clean_rows", float(clean_rows))],
        ["step", "seconds_or_value"]), "spark_timings")

print("\n==== Spark timings (copy into the comparison table) ====")
for n, s in timings:
    print(f"{n:35s} {s:8.2f} s")
spark.stop()

Writing /content/04_spark_analytics.py


In [ ]:
!spark-submit --master yarn --deploy-mode client --num-executors 2 --executor-cores 1 --executor-memory 2g --driver-memory 2g /content/04_spark_analytics.py hdfs:///user/hadoop/nyctaxi/extract hdfs:///user/hadoop/nyctaxi 2>&1 | grep -E "TIME|Raw rows|partitions" | tee /content/spark_run.log

Input partitions (≈ one per 128 MB HDFS block): 2
[TIME] Q0 count raw rows: 18.78 s
[TIME] Q0b clean + count (cache fill): 31.16 s
Raw rows: 1,048,575  Clean rows: 1,018,544  Removed: 30,031 (2.86%)
[TIME] Write clean Parquet: 14.87 s
[TIME] Q1 demand by hour: 6.39 s
[TIME] Q2 dow x hour: 4.38 s
[TIME] Q3 payment type: 6.42 s
[TIME] Q4 hotspots: 3.91 s
[TIME] Q5 daily trend: 2.99 s
[TIME] Summary statistics: 20.42 s


### 📸 Screenshot 11 – YARN application list (Hive = MAPREDUCE, Spark = SPARK)

In [ ]:
!yarn application -list -appStates FINISHED 2>/dev/null | head -30

Total number of applications (application-types: [], states: [FINISHED] and tags: []):15
                Application-Id	    Application-Name	    Application-Type	      User	     Queue	             State	       Final-State	       Progress	                       Tracking-URL
application_1790020707418_0001	SELECT COUNT(*) AS total_rows FROM trips_raw (Stage-1)	           MAPREDUCE	      root	   default	          FINISHED	         SUCCEEDED	           100%	http://d21b880bee1d:19888/jobhistory/job/job_1790020707418_0001
application_1790020707418_0002	CREATE TABLE trips_clean STORED AS ORC A...6 (Stage-1)	           MAPREDUCE	      root	   default	          FINISHED	         SUCCEEDED	           100%	http://d21b880bee1d:19888/jobhistory/job/job_1790020707418_0002
application_1790020707418_0003	SELECT pickup_hour,
       COU...pickup_hour (Stage-1)	           MAPREDUCE	      root	   default	          FINISHED	         SUCCEEDED	           100%	http://d21b880bee1d:19888/jobhistory/job/job_1790

## Done
Download `hive_run.log` and `spark_run.log` from the Files panel if you want them. Send your screenshots and these two numbers back to Claude:
1. **Total blocks** for the full CSV (Screenshot 4)
2. The Hive **Time taken** lines (Screenshot 10)

Claude will put everything into the report.